# 15_03 — Convergencia de las cadenas MCMC · Escenario 5

Paso **3a** del ciclo. Este notebook **sólo diagnostica las cadenas**: no
predice, no evalúa y no toca el bloque de prueba. Si algo aquí falla, los
números de `15_04` no significan nada, y por eso son notebooks separados.

| Estadístico | Referencia | Lectura |
|---|---|---|
| ESS | Geyer (1992) | $\gtrsim 400$ ideal, $>100$ aceptable |
| Geweke $z$ | test $z$ entre segmentos | $\lvert z\rvert < 2$ ⇒ no se rechaza |
| $\hat R$ | Gelman-Rubin | $< 1.1$ ⇒ convergencia |

**Una advertencia propia de este modelo.** El PSBPM es una mezcla con etiquetas
intercambiables: dos cadenas pueden describir la misma posterior con los átomos
permutados y arrojar $\hat R$ enorme sin que nada esté mal. Por eso los
diagnósticos se calculan sobre cantidades **invariantes a la permutación** —el
promedio sobre átomos, y los parámetros que no dependen de la etiqueta como
$\pi_j$ y el número de átomos ocupados—. `fit.extraer_traza_variable` hace ese
promedio; no reemplazarlo por una componente fija.

## 1. Imports, rutas y carga de artefactos

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

from model_psbp_fd.pipelines import (
    cargar_datasets_ar, cargar_hiperparametros, cargar_config_evaluacion,
)
from model_psbp_fd.models.pspb_fd_v3 import PSBPPredictor
from model_psbp_fd.fit import (
    tabla_diagnosticos, resumen_convergencia, diagnostico_variable,
    extraer_traza_variable, matriz_pip, contraste_con_verdad,
)
from model_psbp_fd.graphics import (
    plot_global_components, plot_active_clusters,
    plot_convergence_bj, plot_convergence_pj,
)
from model_psbp_fd.utils import get_project_root

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

In [ ]:
PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# [CONFIG] debe coincidir con 15_01 y con psbp_fd_iteracion.m
BASENAME, ESCENARIO_ID, REPLICA_ID = "escenario", 5, 1
EXPERIMENT_ID = f"{BASENAME}_{ESCENARIO_ID}_r{REPLICA_ID:02d}"

PATHS = {
    "raw":          PROJECT_ROOT / "data" / "simulaciones" / "raw" / EXPERIMENT_ID,
    "functional":   PROJECT_ROOT / "data" / "simulaciones" / "processed" / "functional" / EXPERIMENT_ID,
    "predict":      PROJECT_ROOT / "data" / "simulaciones" / "processed" / "predict" / EXPERIMENT_ID,
    "out_report":   PROJECT_ROOT / "reports" / "simulaciones" / EXPERIMENT_ID,
    "out_artefact": PROJECT_ROOT / "artefact" / "simulaciones" / EXPERIMENT_ID,
}
for r in PATHS.values():
    r.mkdir(parents=True, exist_ok=True)
print(f"EXPERIMENT_ID : {EXPERIMENT_ID}")

In [ ]:
dfs_train, manifest = cargar_datasets_ar(PATHS, bloque="train")
hp_json     = cargar_hiperparametros(PATHS)
eval_config = cargar_config_evaluacion(PATHS)

COMPONENT_IDX = manifest["component_idx"]
n_components  = len(COMPONENT_IDX)
cov_names     = manifest["cov_names"]
N_ITER        = int(hp_json["n_iter"])
MCMC_CFG      = hp_json["mcmc_config"]
BURN          = int(MCMC_CFG["burn"])
REPR_CFG      = eval_config.get("representacion", {})

assert len(hp_json["hyperparams_list"]) == n_components, \
    "n_components discrepa entre manifest e hyperparameters.json."
assert int(hp_json.get("escenario_id", ESCENARIO_ID)) == ESCENARIO_ID, \
    "hyperparameters.json fue escrito para otro escenario."

print(f"componentes : {n_components}  → {COMPONENT_IDX}")
print(f"cadenas     : {N_ITER}   ·   MCMC {MCMC_CFG}")
print(f"seed_base   : {hp_json['seed_base']}   esquema: {hp_json['seed_scheme']}")
print(f"\nRepresentación (declarada por 15_01):")
print(f"  M retenidas                     : {REPR_CFG.get('M_retenidas')}")
print(f"  FPC que lleva la dinámica       : {REPR_CFG.get('fpc_con_dinamica')}")
print(f"  ¿retiene la dirección informativa? {REPR_CFG.get('retiene_direccion_informativa')}")
print(f"  ¿es caso nulo?                  : {REPR_CFG.get('es_caso_nulo')}")

## 2. Lectura de las trazas

El `feature_names` guardado en cada `.mat` se verifica contra las columnas del
dataset de entrenamiento: si el muestreador entrenó con otro orden de
covariables, todo lo que sigue estaría mal etiquetado sin dar error.

In [ ]:
def ruta_traza(fpc_idx: int, chain: int) -> Path:
    return PATHS["out_artefact"] / f"chain_fpc_{fpc_idx}_iter{chain:02d}.mat"


def leer_traza(path: Path):
    """.mat de psbp_train → (traces, burn, feature_names)."""
    m = loadmat(str(path))
    claves = ["betajhout", "beta0hout", "tauhout", "alphahout", "psijhout",
              "Gammajhout", "gammajhout", "pijout", "wjout", "osumout", "inEout"]
    traces = {k: np.asarray(m[k], dtype=np.float64) for k in claves}
    for k in ("muout", "N1out", "Nout"):
        traces[k] = np.asarray(m[k], dtype=np.float64).ravel()
    burn = int(np.asarray(m["burn"]).ravel()[0])
    feat = str(np.atleast_1d(m["feature_names"]).ravel()[0]).split(",")
    return traces, burn, feat


class ModeloTraza:
    """Adaptador trazas MATLAB → interfaz que esperan `graphics` y el predictor v3."""

    def __init__(self, traces, burn, feature_names):
        self.traces = traces
        self.feature_names_ = list(feature_names)
        self.burn = int(burn)
        self.n_features_ = int(traces["betajhout"].shape[2])
        self.predictor_ = PSBPPredictor(traces=traces, burn=burn)

    def _diseno(self, df):
        Xp = np.asarray(df.iloc[:, 1:], dtype=float)
        return np.hstack([np.ones((Xp.shape[0], 1)), Xp])

    def predict(self, df, return_std=False):
        return self.predictor_.predict(self._diseno(df), return_std=return_std)

In [ ]:
faltan = [ruta_traza(COMPONENT_IDX[k] + 1, c + 1).name
          for k in range(n_components) for c in range(N_ITER)
          if not ruta_traza(COMPONENT_IDX[k] + 1, c + 1).exists()]
assert not faltan, (f"Faltan {len(faltan)} trazas en {PATHS['out_artefact']}: "
                    f"{faltan}\nEjecuta psbp_fd_iteracion.m antes de continuar.")

models_chains, _meta = {k: {} for k in range(n_components)}, set()
for k in range(n_components):
    fpc = COMPONENT_IDX[k] + 1
    esperado = list(dfs_train[k].columns[1:])
    for c in range(N_ITER):
        traces, burn, feat = leer_traza(ruta_traza(fpc, c + 1))
        assert feat == esperado, (
            f"[k={k} chain={c+1}] feature_names del .mat != columnas del dataset:\n"
            f"  mat  = {feat}\n  train= {esperado}")
        models_chains[k][c] = ModeloTraza(traces, burn, feat)
        _meta.add((traces["betajhout"].shape[0], traces["betajhout"].shape[1], burn))
        print(f"  FPC {fpc}  chain{c+1:02d}  p={len(feat)}  N={traces['betajhout'].shape[1]}  "
              f"nsim={traces['betajhout'].shape[0]}  burn={burn}")

assert len(_meta) == 1, f"Configuraciones MCMC heterogéneas entre trazas: {_meta}."
nsim, N_ATOMOS, BURN = next(iter(_meta))
assert (nsim, N_ATOMOS, BURN) == (MCMC_CFG["nsim"], MCMC_CFG["N"], MCMC_CFG["burn"]), \
    f"Trazas (nsim={nsim}, N={N_ATOMOS}, burn={BURN}) != hyperparameters.json ({MCMC_CFG})."

n_post = nsim - BURN
print(f"\nOK  {n_components} componente(s) x {N_ITER} cadena(s)")
print(f"  {n_post} draws posteriores por cadena → {n_post * N_ITER} por componente")

## 3. Tabla de diagnósticos

`fit.tabla_diagnosticos` calcula ESS, Geweke y $\hat R$ **sin generar figuras**.
Antes ese cálculo vivía dentro de `plot_convergence_*`, de modo que obtener la
tabla obligaba a dibujar; ahora las figuras de §5 lo consumen del mismo módulo.

In [ ]:
diag_df = tabla_diagnosticos(models_chains, BURN, component_idx=COMPONENT_IDX,
                             claves=("betajhout", "pijout"))
diag_df.to_csv(PATHS["out_report"] / "40_diagnosticos_mcmc.csv", index=False)

display(diag_df.style
    .format({"ess_min": "{:.1f}", "ess_mean": "{:.1f}",
             "geweke_max": "{:+.2f}", "rhat": "{:.4f}"})
    .background_gradient(subset=["rhat"], cmap="RdYlGn_r", vmin=1.0, vmax=1.2)
    .background_gradient(subset=["ess_min"], cmap="RdYlGn", vmin=0, vmax=500)
    .set_caption("Diagnósticos MCMC por componente FPCA y variable"))

In [ ]:
res = resumen_convergencia(diag_df)
print("=" * 62)
print(f"  R-hat máximo    : {res['rhat_max']:.4f}   (umbral {res['umbrales']['rhat']})")
print(f"  ESS mínimo      : {res['ess_min']:.1f}     (umbral {res['umbrales']['ess']})")
print(f"  |Geweke| máximo : {res['geweke_max']:.2f}     (umbral {res['umbrales']['geweke']})")
print(f"  variables sin converger: {res['n_no_converge']} de {res['n_variables']}")
print("=" * 62)
print(f"VEREDICTO: {'todas convergen' if res['todo_converge'] else 'REVISAR'}")

for v in res["variables_malas"]:
    fila = diag_df[(diag_df.componente == v["componente"]) &
                   (diag_df.param == v["param"]) &
                   (diag_df.variable == v["variable"])].iloc[0]
    print(f"  · FPC {v['componente']} {v['param']} {v['variable']}: "
          f"R-hat={fila['rhat']:.3f} ESS={fila['ess_min']:.0f} G={fila['geweke_max']:+.2f}")

## 4. Ocupación de la mezcla

`N1out` es $\max_i S_i$: cuántos átomos están efectivamente ocupados en cada
iteración. Es el diagnóstico que decide si el truncamiento en $N$ átomos es
suficiente y suele ser **lo último en mezclar**, de modo que conviene mirarlo
antes de fijar el `burn`. Si la traza se pega contra $N$, el truncamiento está
mordiendo y hay que subirlo.

In [ ]:
fig, axes = plt.subplots(n_components, 2, figsize=(13, 2.6 * n_components),
                         squeeze=False, gridspec_kw={"width_ratios": [3, 1]})
resumen_ocupacion = []

for k in range(n_components):
    ax_tr, ax_hi = axes[k]
    for c in sorted(models_chains[k]):
        n1 = models_chains[k][c].traces["N1out"]
        ax_tr.plot(np.arange(len(n1)), n1, lw=0.7, alpha=0.75, label=f"cadena {c+1}")
        ax_hi.hist(n1[BURN:], bins=np.arange(0.5, N_ATOMOS + 1.5), alpha=0.55,
                   orientation="horizontal")
        post = n1[BURN:]
        resumen_ocupacion.append({
            "FPC": COMPONENT_IDX[k] + 1, "cadena": c + 1,
            "media": post.mean(), "max": int(post.max()),
            "p99": float(np.quantile(post, 0.99)), "N_trunc": N_ATOMOS,
            "toca_truncamiento": bool(post.max() >= N_ATOMOS),
        })
    ax_tr.axvline(BURN, color="k", ls="--", lw=1, alpha=0.7)
    ax_tr.axhline(N_ATOMOS, color="#c0392b", ls=":", lw=1.2)
    ax_tr.set_ylabel(f"FPC {COMPONENT_IDX[k]+1}\nátomos ocupados")
    ax_tr.set_ylim(0, N_ATOMOS + 1)
    ax_hi.set_ylim(0, N_ATOMOS + 1); ax_hi.set_xlabel("frec. post-burn")
    if k == 0:
        ax_tr.legend(fontsize=8, ncol=3)
        ax_tr.text(BURN, N_ATOMOS, " burn", fontsize=8, va="top")
axes[-1, 0].set_xlabel("iteración")
fig.suptitle(f"Ocupación de la mezcla — truncamiento N={N_ATOMOS}", fontsize=12)
fig.tight_layout()
fig.savefig(PATHS["out_report"] / "41_ocupacion_mezcla.png", dpi=150, bbox_inches="tight")
plt.show()

ocup_df = pd.DataFrame(resumen_ocupacion)
ocup_df.to_csv(PATHS["out_report"] / "41_ocupacion_mezcla.csv", index=False)
display(ocup_df.style.format({"media": "{:.2f}", "p99": "{:.1f}"})
        .set_caption("Átomos ocupados post-calentamiento"))
if ocup_df["toca_truncamiento"].any():
    print("\n[AVISO] Alguna cadena alcanza N con un generador GAUSSIANO: es "
          "inesperado.\n  Revisar los datos antes que el truncamiento.")
else:
    print(f"\nOK  Ninguna cadena alcanza N={N_ATOMOS}: el truncamiento no restringe.")

_ocup_media = ocup_df["media"].mean()
print(f"\nátomos ocupados, media global = {_ocup_media:.2f}")
if _ocup_media < 2.0:
    print("   CORRECTO. La condicional verdadera es una única normal y el "
          "modelo lo\n   reconoce. Debería ser la ocupación más baja de las "
          "seis corridas: anotarla\n   para el contraste con las 12, 13 y 14.")
else:
    print("   El modelo ocupa más de un átomo pese a que la condicional "
          "verdadera es una\n   única normal: está gastando flexibilidad en "
          "ruido. No invalida la corrida,\n   pero cuesta eficiencia y hay que "
          "reportarlo.")

## 5. Figuras de convergencia por parámetro

In [ ]:
for k in models_chains:
    plot_global_components(
        models_chains, k, BURN, N_ITER,
        title_prefix=f"FPC {COMPONENT_IDX[k]+1}",
        save_path=str(PATHS["out_report"] / f"42_global_k{COMPONENT_IDX[k]+1}.png"))
    plt.show()
    plot_active_clusters(
        models_chains, k, BURN, N_ITER,
        title_prefix=f"FPC {COMPONENT_IDX[k]+1}",
        save_path=str(PATHS["out_report"] / f"43_clusters_k{COMPONENT_IDX[k]+1}.png"))
    plt.show()

In [ ]:
for k in models_chains:
    feat = models_chains[k][0].feature_names_
    plot_convergence_bj(models_chains, k, BURN, N_ITER, feature_names=feat,
        title_prefix=f"FPC {COMPONENT_IDX[k]+1}",
        save_path=str(PATHS["out_report"] / f"44_conv_bj_k{COMPONENT_IDX[k]+1}.png"))
    plt.show()
    plot_convergence_pj(models_chains, k, BURN, N_ITER, feature_names=feat,
        title_prefix=f"FPC {COMPONENT_IDX[k]+1}",
        save_path=str(PATHS["out_report"] / f"45_conv_pj_k{COMPONENT_IDX[k]+1}.png"))
    plt.show()

### 6.1 Contraste con la estructura del generador

En el Escenario 5 el operador $\Psi$ es lineal y actúa sobre **toda** la curva,
de modo que en principio toda FPC rezagada es activa. La estructura verdadera
se declara explícitamente aquí porque cambia entre escenarios: en el 5 sólo la
componente subordinada lo es, y en el 3 la actividad depende del régimen.

In [ ]:
pip_df = matriz_pip(models_chains, BURN, component_idx=COMPONENT_IDX)
pip_df.to_csv(PATHS["out_report"] / "46_pip.csv")

cols_pip = [c for c in pip_df.columns if not c.endswith("_sd")]
display(pip_df.style
    .background_gradient(subset=cols_pip, cmap="RdYlGn", vmin=0, vmax=1)
    .format("{:.3f}")
    .set_caption("P(gamma_j = 1 | datos) — inclusión global, media entre cadenas ± sd"))

_sd_max = pip_df[[c for c in pip_df.columns if c.endswith("_sd")]].to_numpy().max()
print(f"dispersión máxima entre cadenas: {_sd_max:.3f}"
      + ("   [AVISO] las cadenas discrepan sobre la selección" if _sd_max > 0.15
         else "   OK  las cadenas coinciden"))

### 6.1 Contraste con la estructura del generador

`VERDAD` **no se escribe a mano**: se deriva de la tabla de alineación que
`15_01 §4` persistió, que a su vez sale de `internos["coeficientes_ar"]`. La
regla es directa:

- las $J$ recursiones del generador son **escalares e independientes**, de modo
  que ninguna covariable **cruzada** es activa, en ninguna componente;
- para la componente FPCA alineada con una componente de Fourier de
  $\varphi_j\neq 0$, la única covariable activa es **su propio rezago**;
- para todas las demás, el conjunto activo es **vacío**.

Nótese que el prior $E[\pi]=0.90$ sobre el propio rezago está **mal orientado en
la mayoría de las componentes** —empuja a incluir una covariable cuyo
coeficiente verdadero es cero exacto—, igual que en la corrida 12. Que el modelo
lo desmienta ahí y lo confirme en la componente correcta es la prueba del eje 3
en este escenario.

In [ ]:
# VERDAD derivada del artefacto, no de la memoria.
_csv_alin = PATHS["out_report"] / "10_alineacion_fpca_generador.csv"
assert _csv_alin.exists(), (
    f"Falta {_csv_alin.name}: regenerar 15_01, que lo escribe en §4. "
    "VERDAD no se declara a mano en este escenario.")
alineacion_df = pd.read_csv(_csv_alin)

# Sólo las componentes efectivamente retenidas entran en el modelo.
_ret = alineacion_df[alineacion_df["retenida"]].set_index("fpc")
assert sorted(_ret.index.tolist()) == sorted([i + 1 for i in COMPONENT_IDX]), \
    ("Las componentes retenidas del CSV no coinciden con COMPONENT_IDX del "
     "manifest: los artefactos son de corridas distintas.")

VERDAD = {}
for k in range(n_components):
    fpc = COMPONENT_IDX[k] + 1
    phi = float(_ret.loc[fpc, "phi_generador"])
    # Recursiones escalares e independientes: sólo el propio rezago puede ser
    # activo, y sólo si su phi verdadero es distinto de cero.
    VERDAD[f"FPC {fpc}"] = [f"fpc_{fpc}_lag1"] if phi != 0.0 else []

print("Conjunto activo verdadero, derivado de internos['coeficientes_ar']:")
for nombre, activas in VERDAD.items():
    _f = int(nombre.split()[1])
    print(f"  {nombre:<10} phi={float(_ret.loc[_f, 'phi_generador']):+.2f}  "
          f"(Fourier {int(_ret.loc[_f, 'fourier_alineada'])})  → {activas}")

_n_activas = sum(len(v) for v in VERDAD.values())
_n_total   = n_components * len(cov_names)
print(f"\ncovariables activas: {_n_activas} de {_n_total} "
      f"({_n_activas / _n_total:.1%})")
assert _n_activas > 0, (
    "Ninguna covariable es activa: la componente con dinámica quedó fuera del "
    "truncamiento. El escenario MUERDE, y en ese caso la lectura del eje 3 es "
    "la de la corrida 12: la cifra reportable es `pip_media_inactivas`, y ni "
    "sensibilidad ni AUC están definidas. Comentar la línea de abajo y seguir.")

In [ ]:
contraste = contraste_con_verdad(pip_df, VERDAD, umbral=0.5)
if not contraste.empty:
    contraste.to_csv(PATHS["out_report"] / "47_contraste_pip.csv")
    display(contraste.style.format("{:.3f}", subset=[
        c for c in contraste.columns if contraste[c].dtype.kind == "f"])
        .set_caption("Selección de variables vs estructura del generador (umbral 0.5)"))
    print("Con conjunto activo pequeño y NO vacío, sensibilidad, especificidad y "
          "AUC están\ntodas bien definidas: es el único escenario del estudio en "
          "que las tres son\ninformativas a la vez.")

# La comparación que decide: PIP del propio rezago en la componente con dinámica
# contra la de las componentes sin dinámica, donde el prior es el mismo (0.90).
print("\nPIP del PROPIO rezago, componente a componente:")
for k in range(n_components):
    fpc  = COMPONENT_IDX[k] + 1
    fila = f"FPC {fpc}"
    col  = f"fpc_{fpc}_lag1"
    if fila in pip_df.index and col in pip_df.columns:
        _v   = float(pip_df.loc[fila, col])
        _phi = float(_ret.loc[fpc, "phi_generador"])
        marca = ("  <- ACTIVA de verdad (phi != 0)" if _phi != 0
                 else "  (phi = 0 exacto: el prior 0.90 está mal orientado)")
        print(f"  {fila:<10} PIP={_v:.3f}{marca}")

print("\nLo que hay que ver: PIP alta donde phi != 0 y PIP claramente MENOR "
      "donde phi = 0,\npese a que el prior es 0.90 en ambos casos. Si todas "
      "fueran altas, el modelo\nestaría siguiendo al prior y no a los datos.")

## 7. Verificación muestreador ↔ predictor

`inEout` es $E[y\mid x]$ dentro de muestra tal como la calcula MATLAB en cada
iteración. Debe coincidir con lo que el predictor de Python reconstruye a
partir de las mismas trazas. Es la única prueba directa de que ambos lados
interpretan las trazas igual, y falla ruidosamente si el contrato se rompe.

In [ ]:
print("Verificación inEout vs predict sobre el bloque de entrenamiento:")
ok = True
for k in models_chains:
    fpc  = COMPONENT_IDX[k] + 1
    _phi = float(_ret.loc[fpc, "phi_generador"])
    for c in sorted(models_chains[k]):
        m = models_chains[k][c]
        mu_tr = m.predict(dfs_train[k])
        inE   = m.traces["inEout"][m.burn:].mean(axis=0)
        corr  = float(np.corrcoef(inE, mu_tr)[0, 1])
        dif   = float(np.abs(inE - mu_tr).max())
        esc   = float(np.std(dfs_train[k].iloc[:, 0]))
        # El criterio que manda es |dif|max; la correlación sólo es exigible
        # donde la media condicional verdadera varía (phi != 0).
        ok_k = (dif < 0.05 * esc) and (corr > 0.999 or _phi == 0.0)
        ok &= ok_k
        print(f"  FPC {fpc} cadena{c+1}: corr={corr:.6f}  |dif|max={dif:.4f}  "
              f"({dif/esc:.2%} de sd(y))"
              + ("" if ok_k else "   [REVISAR]")
              + ("   (phi=0: corr no exigible)" if _phi == 0.0 else ""))

print(f"\n{'contrato sampler-predictor verificado' if ok else 'el contrato FALLA'}")
print(f"\n{'='*62}\nSi el veredicto de §3 y esta verificación son correctos, "
      f"continuar con\n15_04_evaluacion.\n{'='*62}")